# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa — Exploration with `mlcroissant`
This notebook demonstrates how to explore the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset is defined by a [Croissant schema](https://mlcommons.org/croissant/) and can be found at the following URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

We will load dataset metadata, examine available record sets and fields, and perform some exploratory data analysis.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant>=1.0

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for clarity

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access key metadata fields (no subscripting, use .attributes)
print(f"Dataset title: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In Croissant, the main tables (record sets) are referenced by their `@id`. Fields (columns) are also referenced by their `@id`.

In [ ]:
# List all record sets and their fields
record_sets = dataset.metadata.recordSets
print("RecordSets in this dataset:")
for rs in record_sets:
    print(f"  - @id: {rs.id} | name: {rs.name}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      · @id: {field.id} | name: {field.name} | dataType: {field.dataType}")
    print("")

# Choose a main record set to work with (for demonstration, pick the first one)
main_record_set_id = record_sets[0].id if record_sets else None
print(f"Will use main record set `@id`: {main_record_set_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Each record set can be processed via its `@id`.

In [ ]:
# List all record set @ids to extract
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    # Load records
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"  -> No records found for {record_set_id}\n")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Rows: {len(df)}\n")
    print(df.head(2))
    print("")

# Example: Display columns in the main record set
if main_record_set_id in dataframes:
    print(f"Columns in main record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())


## 4. Exploratory Data Analysis (EDA)
Let's perform basic EDA: filter, normalize, group, and summarize with explicit `@id` references for fields.

Suppose we want to analyze the PHQ-9 score field (Patient Health Questionnaire 9-item) which, in this dataset, is named and referenced by its `@id`.

*Adjust field `@id` variables according to the overview above for your experiment.*

In [ ]:
# -- Please update these @ids if record set structure changes in the dataset --
# For demonstration, select likely columns: phq9_score_field_id and group_field_id

# Use the main record set for EDA
rs_id = main_record_set_id
df = dataframes.get(rs_id)

# Identify PHQ-9 column by its @id from the schema
phq9_field_id = None
education_field_id = None
for field in record_sets[0].fields:
    if 'phq' in field.name.lower() and ('score' in field.name.lower() or field.dataType=='Integer'):
        phq9_field_id = field.id
    if 'education' in field.name.lower():
        education_field_id = field.id
print(f"Using PHQ-9 field @id: {phq9_field_id}")
print(f"Using Education field @id: {education_field_id}")

# Filter for respondents with PHQ-9 > 10 (moderate to severe depression)
threshold = 10
if phq9_field_id in df.columns:
    filtered_df = df[df[phq9_field_id].astype(float) > threshold].copy()
    print(f"Filtered records where {phq9_field_id} > {threshold}:")
    display(filtered_df[[phq9_field_id]].head())

    # Normalize PHQ-9 (z-score)
    filtered_df[f"{phq9_field_id}_normalized"] = (filtered_df[phq9_field_id].astype(float) - filtered_df[phq9_field_id].astype(float).mean()) / filtered_df[phq9_field_id].astype(float).std()
    print(f"Normalized {phq9_field_id} for filtered records:")
    display(filtered_df[[phq9_field_id, f"{phq9_field_id}_normalized"]].head())

    # Group by education level if present
    group_field = education_field_id  # You may choose another grouping field
    if group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[phq9_field_id].mean().reset_index().sort_values(phq9_field_id, ascending=False)
        print(f"Mean PHQ-9 by {group_field}:")
        display(grouped_df)
else:
    print(f"Could not find PHQ-9 field (@id: {phq9_field_id}) in DataFrame columns. Available columns: {df.columns.tolist()}")

## 5. Visualization
Visualize the distribution of the PHQ-9 scores and group means by education.

We'll use `matplotlib` and `seaborn` for convenience.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if phq9_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[phq9_field_id].astype(float), kde=True, bins=10, color='skyblue')
    plt.title(f"Distribution of PHQ-9 Scores ({phq9_field_id})")
    plt.xlabel("PHQ-9 Score")
    plt.ylabel("Number of Respondents")
    plt.show()

    if education_field_id in df.columns:
        plt.figure(figsize=(10, 6))
        means = df.groupby(education_field_id)[phq9_field_id].mean().sort_values(ascending=False)
        sns.barplot(x=means.values, y=means.index, palette='Blues_d')
        plt.title(f"Mean PHQ-9 by Education Level ({education_field_id})")
        plt.xlabel("Mean PHQ-9 Score")
        plt.ylabel("Education Level (@id)")
        plt.show()

## 6. Conclusion
In this notebook, we:

1. Loaded the FAIR² Mental Health Survey Dataset using its Croissant schema via `mlcroissant`.
2. Explored available record sets and fields, referencing all entities by their `@id` for reproducibility.
3. Loaded records into pandas DataFrames for flexible analysis.
4. Performed EDA, including filtering and normalizing PHQ-9 scores (by field `@id`) and grouping by education level.
5. Generated visualizations of PHQ-9 distribution and group means.

**Next Steps**: You can repeat this workflow for other fields (e.g., GAD-7, PSQ) or link to other record sets. Adjust field and record set `@id`s as needed for other analyses.